In [7]:


import os
import sys
import json
import time
import math
import pickle
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
import mlflow.pyfunc
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.datasets import (
    load_iris, load_diabetes, load_digits, load_linnerud,
    load_wine, load_breast_cancer, make_classification, make_regression
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor

warnings.filterwarnings("ignore")


def find_project_root(start_path=None):
    """Find the mlflow_for_ml_dev folder from wherever Jupyter is started."""
    start = Path(start_path or Path.cwd()).resolve()
    for parent in [start] + list(start.parents):
        if parent.name == "mlflow_for_ml_dev" and (parent / "notebooks").exists():
            return parent
        if (parent / "datasets").exists() and (parent / "mlruns").exists() and (parent / "notebooks").exists():
            return parent
    # Fallback: if user started Jupyter inside notebooks subfolder
    for parent in [start] + list(start.parents):
        if parent.name == "notebooks":
            return parent.parent
    return start

PROJECT_ROOT = find_project_root()
DATASETS_DIR = PROJECT_ROOT / "datasets"
MLRUNS_DIR = PROJECT_ROOT / "mlruns"
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "artifacts_generated"

for folder in [DATASETS_DIR, MLRUNS_DIR, MODELS_DIR, ARTIFACTS_DIR, MLRUNS_DIR / ".trash"]:
    folder.mkdir(parents=True, exist_ok=True)

# IMPORTANT: Every notebook logs to the same mlruns folder inside this project.
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
client = MlflowClient()

print("Project root      :", PROJECT_ROOT)
print("Datasets folder   :", DATASETS_DIR)
print("MLflow runs folder:", MLRUNS_DIR)
print("Models folder     :", MODELS_DIR)
print("Tracking URI      :", mlflow.get_tracking_uri())


Project root      : /Users/giridharsripathi/Documents/training/cgi_mlflow_new
Datasets folder   : /Users/giridharsripathi/Documents/training/cgi_mlflow_new/datasets
MLflow runs folder: /Users/giridharsripathi/Documents/training/cgi_mlflow_new/mlruns
Models folder     : /Users/giridharsripathi/Documents/training/cgi_mlflow_new/models
Tracking URI      : file:///Users/giridharsripathi/Documents/training/cgi_mlflow_new/mlruns


In [8]:
# ============================================================
# Read regression dataset
# ============================================================

# This dataset contains house details and a target column called price.
# The goal is to predict the house price.
df = pd.read_csv("../../datasets/house_price_1000.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (1000, 11)


,area_sqft,bedrooms,bathrooms,property_age,distance_city_km,school_rating,parking_slots,location,property_type,furnishing,price
0,4831,3,3,31,7.10,8.9,3,Good,Villa,Unfurnished,29964929.13
1,1623,3,2,24,15.26,9.4,0,Good,Apartment,Fully-Furnished,13438472.98
2,4654,4,1,41,20.84,5.9,0,Good,Apartment,Fully-Furnished,25618663.81
3,4665,4,1,16,10.93,9.6,1,Good,Apartment,Fully-Furnished,27933641.19
4,1875,4,4,14,20.38,9.0,0,Developing,Apartment,Unfurnished,13841900.43


In [9]:
# ============================================================
# Basic EDA
# ============================================================

# info() shows column names, non-null count, and data types.
df.info()

# Check missing values column-wise.
print("\nMissing values:")
print(df.isnull().sum())

# Summary of numerical columns.
display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   area_sqft         1000 non-null   int64  
 1   bedrooms          1000 non-null   int64  
 2   bathrooms         1000 non-null   int64  
 3   property_age      1000 non-null   int64  
 4   distance_city_km  1000 non-null   float64
 5   school_rating     1000 non-null   float64
 6   parking_slots     1000 non-null   int64  
 7   location          1000 non-null   object 
 8   property_type     1000 non-null   object 
 9   furnishing        1000 non-null   object 
 10  price             1000 non-null   float64
dtypes: float64(3), int64(5), object(3)
memory usage: 86.1+ KB

Missing values:
area_sqft           0
bedrooms            0
bathrooms           0
property_age        0
distance_city_km    0
school_rating       0
parking_slots       0
location            0
property_type       0
furni

,area_sqft,bedrooms,bathrooms,property_age,distance_city_km,school_rating,parking_slots,price
count,1000.000000,1000.000000,1000.0000,1000.000000,1000.000000,1000.000000,1000.000000,1.000000e+03
mean,2775.344000,3.048000,2.4210,23.241000,20.520340,5.478100,1.559000,1.744036e+07
std,1311.395156,1.438668,1.1274,12.970858,11.341427,2.548106,1.120614,7.165398e+06
min,451.000000,1.000000,1.0000,0.000000,1.020000,1.000000,0.000000,9.000000e+05
25%,1620.500000,2.000000,1.0000,12.000000,10.595000,3.400000,1.000000,1.154669e+07
50%,2705.000000,3.000000,2.0000,24.000000,20.200000,5.500000,2.000000,1.744935e+07
75%,3957.000000,4.000000,3.0000,34.000000,30.537500,7.600000,3.000000,2.341151e+07
max,4995.000000,5.000000,4.0000,44.000000,39.970000,10.000000,3.000000,3.525574e+07


In [10]:
# ============================================================
# Prepare regression data
# ============================================================

# Separate input features and target.
X = df.drop("price", axis=1)
y = df["price"]

# Convert categorical columns into numeric columns.
X = pd.get_dummies(X, drop_first=True)

# Split data into training and testing data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (800, 14)
X_test : (200, 14)


In [11]:
# ============================================================
# Train ElasticNet and track regression metrics
# ============================================================

mlflow.set_experiment("day2_regression_tracking")

alpha = 0.1
l1_ratio = 0.5

with mlflow.start_run(run_name="elasticnet_regression"):

    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    mlflow.log_param("model_name", "ElasticNet")
    mlflow.log_param("alpha", alpha)
    mlflow.log_param("l1_ratio", l1_ratio)

    mlflow.log_metric("mae", mae)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2_score", r2)

    mlflow.sklearn.log_model(model, name="model")

    print("ElasticNet run logged.")

TypeError: log_model() got an unexpected keyword argument 'name'

In [ ]:
# ============================================================
# Evaluate regression model
# ============================================================

# MAE:
# Average absolute difference between actual and predicted value.
mae = mean_absolute_error(y_test, y_pred)

# MSE:
# Average squared error. Penalizes large mistakes more.
mse = mean_squared_error(y_test, y_pred)

# RMSE:
# Square root of MSE. Same unit as target value.
rmse = np.sqrt(mse)

# R2:
# How much variance in target is explained by model.
r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 2))
print("MSE :", round(mse, 2))
print("RMSE:", round(rmse, 2))
print("R2  :", round(r2, 4))

MAE : 757887.54
MSE : 908385784948.03
RMSE: 953092.75
R2  : 0.9802
